In [ ]:
import requests
from bs4 import BeautifulSoup
from typing import Tuple, List, Optional, Dict
from urllib.parse import urljoin
import urllib.parse
session = requests.Session()
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'Accept-Language': 'vi-VN,vi;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'DNT': '1',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'none',
    'Sec-Fetch-User': '?1',
    'Cache-Control': 'max-age=0',
}



def extract_text(entry):
    all_paragraphs = [p.get_text(strip=True) for p in entry.select('p')]
    full_text = '\n'.join(all_paragraphs)
    return full_text


def scrape_luatvietnam(url):
    response = session.get(url, headers=headers)

    soup = BeautifulSoup(response.content, 'lxml')

    main = soup.select_one("main.main")
    the_article = main.select_one("div.section div.main-content div.section-an-le div.content-left article.the-article")
    entry_hoidap_ls = the_article.select_one("div.entry-title-ls div.entry-hoidap-ls")
    entry_hoidap = the_article.select_one("div.entry-hoi-dap div.the-article-body.entry")
    question =""
    answer =""
    if entry_hoidap_ls:
        question = extract_text(entry_hoidap_ls)
    if entry_hoidap:
        answer = extract_text(entry_hoidap)
    else:
        print("No hoidap article")
    return question, answer



In [ ]:
url="https://luatvietnam.vn/luat-su-tu-van/tai-san-bo-me-mua-cho-con-duoi-16-tuoi-thi-con-co-quyen-ban-khong-149956-faqs.html"
question,answer = scrape_luatvietnam(url)
print(question)
print("-*20")
print(answer)

In [ ]:
def scrape_url(url):
    response = session.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'lxml')
    main = soup.select_one("main.main")
    table = main.select_one("table.table-boderd.table-hoi-dap")
    # Lấy TẤT CẢ <a> trong <tr>
    links = [a['href'] for a in soup.select("tr h3 a")]
    return links

In [ ]:
links = scrape_url("https://luatvietnam.vn/luat-su-tu-van/dat-dai-nha-o-3.html?pSize=50&page=9")
print(links)

In [ ]:
import os
import csv
from typing import Dict, Any

def write_csv_row(filename: str, fieldnames: List[str], row: Dict[str, Any]):
    """Ghi 1 dòng vào CSV - mode='a' tự động"""
    file_exists = os.path.isfile(filename)

    with open(filename, 'a', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)

        if not file_exists:  # Chỉ ghi header lần đầu
            writer.writeheader()

        writer.writerow(row)
        f.flush()
count = 0
error = 0
for i in range(3,9):
    url =f"https://luatvietnam.vn/luat-su-tu-van/dat-dai-nha-o-3.html?pSize=50&page={i}"
    links = scrape_url(url)
    for link in links:
        p_link = urljoin("https://luatvietnam.vn", link)
        print(f"scrape_luatvietnam({p_link})")
        question, answer = scrape_luatvietnam(p_link)
        if question and answer:
            count += 1
            print(f"scraped {count}")
            write_csv_row("qna_streaming.csv", ['question', 'answer', 'link'], {
                "question": question,
                "answer": answer,
                "link": p_link,
            })
        else:
            error += 1
            print(f"error {error}")
            write_csv_row("error_links.csv", ['link'], {
                'link': p_link,
            })
print("done")
print(f"scraped {count}")
print(f"error {error}")


In [ ]:
import pandas as pd
from openpyxl.styles import Alignment, Font  # ← Alignment ở đây!
from openpyxl import load_workbook

# Đọc CSV của bạn
df = pd.read_csv("qna_streaming.csv")

# Tạo Excel với WRAP TEXT
excel_file = "qna_wrapped.xlsx"

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Ghi data
    df.to_excel(writer, sheet_name='Q&A', index=False)

    # Lấy worksheet để format
    worksheet = writer.sheets['Q&A']

    # ✅ Alignment(wrap_text=True) - Tự xuống hàng!
    for row in worksheet.iter_rows(min_row=1):  # Tất cả rows
        for cell in row:
            cell.alignment = Alignment(
                wrap_text=True,      # ✅ Xuống hàng tự động
                vertical='top',      # ✅ Căn trên
                horizontal='left'    # ✅ Căn trái
            )

    # Bold header
    for cell in worksheet[1]:  # Row 1 = header
        cell.font = Font(bold=True)
        cell.alignment = Alignment(wrap_text=True, horizontal='center')

print(f"✅ Tạo xong: {excel_file}")
print("📝 Text tiếng Việt + \\n sẽ tự xuống hàng!")

In [ ]:
!pip install pandas openpyxl

In [ ]:
import pandas as pd
import sys
import os
from openpyxl.styles import Alignment, Font
import openpyxl.utils
from pathlib import Path

def csv_to_excel_wrapped_perfect(csv_input: str, excel_output: str = None):
    """
    Chuyển CSV tiếng Việt có \n → Excel wrap text trong 1 cell
    """

    # Kiểm tra file input
    if not os.path.exists(csv_input):
        print(f"❌ File không tồn tại: {csv_input}")
        return

    # Tự tạo tên output
    if excel_output is None:
        excel_output = str(Path(csv_input).with_suffix('.xlsx'))

    print(f"📥 Đọc CSV: {csv_input}")

    # Đọc CSV - pandas tự handle \n + UTF-8
    df = pd.read_csv(csv_input, encoding='utf-8-sig', dtype=str)
    print(f"📊 Data: {len(df)} dòng, {len(df.columns)} cột")

    # Tạo Excel PERFECT
    with pd.ExcelWriter(excel_output, engine='openpyxl') as writer:
        # Ghi data
        df.to_excel(writer, sheet_name='Data', index=False)
        ws = writer.sheets['Data']

        # 🔥 STEP 1: WRAP TEXT tất cả cells
        for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
            for cell in row:
                cell.alignment = Alignment(
                    wrap_text=True,       # ✅ Xuống hàng trong cell
                    vertical='top',       # ✅ Căn đầu cell
                    horizontal='left'
                )

        # 🔥 STEP 2: BOLD + CENTER header
        for cell in ws[1]:  # Row header
            cell.font = Font(bold=True, size=12)
            cell.alignment = Alignment(
                wrap_text=True,
                horizontal='center',
                vertical='center'
            )

        # 🔥 STEP 3: AUTO COLUMN WIDTH
        for col_idx, column in enumerate(ws.columns, 1):
            col_letter = openpyxl.utils.get_column_letter(col_idx)
            max_length = 0

            for cell in column:
                if cell.value:
                    cell_length = len(str(cell.value))
                    max_length = max(max_length, cell_length)

            # Width tối ưu
            ws.column_dimensions[col_letter].width = min(max_length + 2, 80)

        # 🔥 STEP 4: AUTO ROW HEIGHT (CRITICAL!)
        for row_num in range(1, ws.max_row + 1):
            row_height = 20  # Min height

            for col in ws.iter_cols(min_col=1, max_col=ws.max_column, min_row=row_num, max_row=row_num):
                cell = col[0]
                if cell.value:
                    # Ước lượng height theo text length
                    text_len = len(str(cell.value))
                    height = max(20, (text_len // 50) * 18 + 20)
                    row_height = max(row_height, height)

            ws.row_dimensions[row_num].height = row_height

        # 🔥 STEP 5: Freeze header + dễ nhìn
        ws.freeze_panes = 'A2'
        ws.sheet_view.showGridLines = True

    print(f"✅ HOÀN THÀNH!")
    print(f"📤 Excel: {excel_output}")
    print(f"🎉 Text \\n đã xuống hàng trong CÙNG 1 CELL!")
    print(f"📏 Auto width/height - Sẵn sàng dùng!")

In [ ]:
csv_to_excel_wrapped_perfect("qna_streaming.csv", "output.xlsx")

In [ ]:
import pandas as pd

# Đọc file CSV
df = pd.read_csv("qna_streaming.csv", encoding="utf-8")

# Thay thế "/n" thành xuống dòng thực sự "\n"
df = df.applymap(lambda x: x.replace("/n", "\n") if isinstance(x, str) else x)

# Xuất ra Excel
df.to_excel("output.xlsx", index=False)

print("Đã chuyển đổi xong!")

In [ ]:
import os

from groq import Groq

client = Groq(

)
answer = """Trả lời:
-Luật hộ tịch 2008(sửa đổi, bổ sung năm 2014)
-Luật đất đai 2013;
-Luật nhà ở 2014;
-Nghị định 43/2014/NĐ-CPhướng dẫn thi hành Luật đất đai;
Trước hết, theo thông tin anh cung cấp, anh có quốc tịch Việt Nam và đã sinh sống thường trú tại Canada. Căn cứ vào khoản 3, Điều 3 Luật Quốc tịch Việt Nam 2008: “3. Người Việt Nam định cư ở nước ngoài là công dân Việt Nam và người gốc Việt Nam cư trú, sinh sống lâu dài ở nước ngoài”, như vậy anh thuộc đối tượng người Việt Nam định cư ở nước ngoài.
Về vấn đề nhận quyền sử dụng đất đối với người Việt Nam định cư ở nước ngoài, điểm đ, Khoản 1, Điều 169 Luật Đất đai 2013 quy định như sau:
đ) Người Việt Nam định cư ở nước ngoài thuộc diện được sở hữu nhà ở tại Việt Nam theo quy định của pháp luật về nhà ở được nhận chuyển quyền sử dụng đất ở thông qua hình thức mua, thuê mua, nhận thừa kế, nhận tặng cho nhà ở gắn liền với quyền sử dụng đất ở hoặc được nhận quyền sử dụng đất ở trong các dự án phát triển nhà ở;
Khoản 1, Điều 186 Luật Đất đai 2013 cũng có quy định về Quyền và nghĩa vụ về sử dụng đất ở của người Việt Nam định cư ở nước ngoài được sở hữu nhà ở tại Việt Nam như sau:
1. Người Việt Nam định cư ở nước ngoài thuộc các đối tượng có quyền sở hữu nhà ở theo quy định của pháp luật về nhà ở thì có quyền sở hữu nhà ở gắn liền với quyền sử dụng đất ở tại Việt Nam.
Điều 7 Luật Nhà ở năm 2014 quy định người Việt Nam định cư ở nước ngoài thuộc một trong các đối tượng được sở hữu nhà ở tại Việt Nam. Theo Điều 8 Luật Nhà ở 2014, điều kiện để người Việt Nam định cư ở nước ngoài được công nhận quyền sở hữu nhà ở tại Việt Nam gồm:
Thứ nhất,được phép nhập cảnh vào Việt Nam
Thứ hai,có nhà ở hợp pháp thông qua hình thức mua, thuê mua nhà ở thương mại của doanh nghiệp, hợp tác xã kinh doanh bất động sản (sau đây gọi chung là doanh nghiệp kinh doanh bất động sản); mua, nhận tặng cho, nhận đổi, nhận thừa kế nhà ở của hộ gia đình, cá nhân; nhận chuyển nhượng quyền sử dụng đất ở trong dự án đầu tư xây dựng nhà ở thương mại được phép bán nền để tự tổ chức xây dựng nhà ở theo quy định của pháp luật”.
Như vậy, căn cứ vào các quy định nêu trên, người Việt Nam định cư ở nước ngoài chỉ có thể được nhận chuyển nhượng trực tiếp quyền sử dụng đất trong trong các dự án đầu tư xây dựng nhà ở thương mại được phép bán nền để tự tổ chức xây dựng nhà ở theo quy định của pháp luật. Đối với các trường hợp khác, pháp luật chỉ cho phép người Việt Nam định cư ở nước ngoài nhận chuyển quyền sử dụng đất ở thông qua hình thức mua, thuê mua, nhận thừa kế, nhận tặng cho nhà ở gắn liền với quyền sử dụng đất ở, tức nhận chuyển quyền sử dụng đất phải gắn liền với nhà ở. Trong cả hai trường hợp nêu trên, người Việt Nam định cư ở nước ngoài bắt buộc phải đáp ứng điều kiện được phép nhập cảnh vào Việt Nam.

Đối với trình tự, thủ tục mua bất động sản thuộc diện được phép chuyển nhượng của người Việt Nam định cư ở nước ngoài tại Việt Nam, Anh có thể thực hiện theo các quy định của pháp luật như sạu:
Bước 1: Công chứng hợp đồng
Hai bên chuyển nhượng và nhận chuyển nhượng đến tổ chức công chứng trên địa bàn tỉnh nơi có đất yêu cầu công chứng hợp đồng chuyển nhượng quyền sử dụng đất. Các giấy tờ cần công chứng bao gồm:
- Phiếu yêu cầu công chứng hợp đồng (theo mẫu).
- Dự thảo hợp đồng chuyển nhượng (nếu có).
- Bản gốc CMND, CCCD, Hộ chiếu của bên chuyển nhượng và bên nhận chuyển nhượng.
- Bản gốc giấy chứng nhận quyền sử dụng đất.
- Bản sao giấy tờ khác có liên quan đến hợp đồng.
Bước 2: Kê khai nghĩa vụ tài chính
Bạn thực hiện kê khai nghĩa vụ tài chính tại Văn phòng đăng ký đất đai. Hồ sơ thực hiện việcsang tên sổ đỏgồm:
- Tờ khai lệ phí trước bạ (02 bản do bên mua ký)
- Tờ khai thuế thu nhập cá nhân (02 bản do bên bán ký.
- Hợp đồng công chứng đã lập (01 bản chính)
- Giấy chứng nhận quyền sử dụng đất (sổ đỏ); quyền sở hữu nhà ở và tài sản gắn liền với đất (01 bản sao có chứng thực của cơ quan có thẩm quyền).
- CMND + Sổ hộ khẩu của cả bên mua và bên bán (01 bản sao có chứng thực của cơ quan có thẩm quyền)
- Đối với trường hợp cho tặng, thừa kế phải có giấy tờ chứng minh quan hệ nhân thân của người cho và người nhận để được miễn thuế thu nhập cá nhân.
Thời hạn có thông báo nộp thuế: 10 ngày. Sau khi có thông báo thì người nộp thuế nộp tiền vào ngân sách nhà nước.
Bước 3: Kê khai hồ sơ sang tên
Hồ sơ sang tên sổ đỏ gồm:
- Đơn đề nghị đăng ký biến động (do bên bán ký); Trong trường hợp có thoả thuận trong hợp đồng về việc bên mua thực hiện thủ tục hành chính thì bên mua có thể ký thay.
- Hợp đồng chuyển nhượng
- Giấy chứng nhận quyền sử dụng đất (sổ đỏ), quyền sở hữu nhà và tài sản gắn liền với đất (bản gốc)
- Giấy nộp tiền vào ngân sách nhà nước (bản gốc)
- Bản sao CMND, CCCD, Hộ chiếu của bên nhận chuyển nhượng.
Bước 4: Nhận kết quả
Sau khi hoàn tất nghĩa vụ tài chính, bạn nộp biên lai cho Văn phòng đăng ký đất đai để nhận Giấy chứng nhận quyền sử dụng đất và sở hữu nhà ở/căn hộ trên đất.
Xem thêm:Việt kiều có được đứng tên Sổ đỏ, Sổ hồng không?
Trên đây là nội dung tư vấn về ""​Người song tịch có được mua đất ở Việt Nam không?"" dựa trên những thông tin mà luật sư đã nhận được. Nếu còn bất kỳ thắc mắc nào liên quan, vui lòng liên hệ 19006192 để được hỗ trợ kịp thời. Xin cảm ơn!"""
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": f"Trích xuất tên các văn bản pháp luật có trong đoạn sau, chỉ trả về các tên văn bản, không trả lời thừa thãi câu trả lời, ngăn cách nhau bằng dấu \ n. Đoạn văn: {answer}",
        }
    ],
    model="llama-3.1-8b-instant",
)

print(chat_completion.choices[0].message.content)

In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("OPENAI_API_KEY not found in environment or .env file.")
client = OpenAI(api_key=api_key)

In [ ]:
import os
import pandas as pd
import asyncio
from openai import AsyncOpenAI
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
import openpyxl

async def process_excel_streaming(
    input_file: str,
    output_file: str,
    input_column: str,
    output_column: str,
    model: str = "gpt-4o-mini"
):
    """
    Xử lý TỪNG DÒNG XLSX → OpenAI → Lưu NGAY XLSX → 0 RAM!
    Input & Output đều là .xlsx ✅
    """

    print(f"🤖 Model: {model} (RẺ NHẤT!)")
    print(f"📥 Input: {input_file}")

    # 🔥 ĐỌC XLSX INPUT
    df_input = pd.read_excel(input_file, engine='openpyxl')
    total_rows = len(df_input)
    print(f"📊 Tổng: {total_rows} dòng")

    # Tạo output file nếu chưa có
    if not os.path.exists(output_file):
        df_input.to_excel(output_file, index=False, engine='openpyxl')

    # 🔥 MỞ OUTPUT XLSX
    wb = load_workbook(output_file)
    ws = wb.active

    # Thêm cột output nếu chưa có
    col_idx = len(df_input.columns) + 1
    if ws.cell(row=1, column=col_idx).value is None:
        ws.cell(row=1, column=col_idx, value=output_column)
        wb.save(output_file)

    print(f"✍️ Ghi kết quả cột: {get_column_letter(col_idx)}")

    # Khởi tạo OpenAI client
    client = AsyncOpenAI()

    processed = 0
    cost_estimate = 0

    # 🔥 XỬ LÝ TỪNG DÒNG TỪ XLSX INPUT
    for line_num, (_, row_data) in enumerate(df_input.iterrows(), 1):
        question = str(row_data.get(input_column, "")).strip()
        if not question:
            print(f"\n[{line_num}/{total_rows}] Bỏ qua: Rỗng")
            continue

        print(f"\n[{line_num}/{total_rows}] Xử lý: {question[:60]}...")

        try:
            # 🔥 GỌI OPENAI gpt-4o-mini ✅
            response = await client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": """
                    Bạn là chuyên gia pháp luật Việt Nam.
                    Nhiệm vụ: Trích xuất TẤT CẢ văn bản pháp luật từ đoạn văn.

                    Định dạng:
                    1. Chỉ danh sách tên, mỗi cái 1 dòng
                    2. Format: [Loại] [Số hiệu] [Năm]
                    3. VD: Luật Đất đai 2013, Nghị định 43/2014/NĐ-CP
                    4. KHÔNG giải thích, KHÔNG lời dẫn
                    5. Không có → chuỗi rỗng
                    """},
                    {"role": "user", "content": f"Trích xuất: {question}"}
                ],
                temperature=0.1
            )

            result = response.choices[0].message.content.strip()
            tokens = response.usage.total_tokens

            # 🔥 SAO CHÉP DÒNG GỐC + KẾT QUẢ
            for col_num, col_name in enumerate(df_input.columns, 1):
                value = row_data[col_name]
                ws.cell(row=line_num + 1, column=col_num, value=value)

            ws.cell(row=line_num + 1, column=col_idx, value=result)

            # 🔥 LƯU NGAY SAU MỖI DÒNG!
            wb.save(output_file)

            processed += 1
            cost_estimate += tokens * 0.00015 / 1_000_000  # Giá gpt-4o-mini chính xác

            print(f"✅ {result[:80]}...")
            print(f"💰 Tokens: {tokens} | Tổng: ${cost_estimate:.4f}")

        except Exception as e:
            print(f"❌ Lỗi dòng {line_num}: {e}")
            ws.cell(row=line_num + 1, column=col_idx, value=f"LỖI: {str(e)}")
            wb.save(output_file)

    wb.close()
    print(f"\n🎉 HOÀN THÀNH {processed}/{total_rows} dòng!")
    print(f"💸 Ước tính chi phí: ${cost_estimate:.4f}")
    print(f"📁 Output: {output_file}")

import nest_asyncio
nest_asyncio.apply()

# Chạy trực tiếp (không cần asyncio.run)
await process_excel_streaming(
    'output.xlsx', 'result.xlsx', 'input', 'output'
)


